# AADMF Ablation Analysis

Publication-quality charts for Evaluation Document (Document 5), Sections 5.2-5.5.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# High-quality figure defaults
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Load CSV produced by ablation runner
candidate_paths = [
    Path('experiments/results/ablation_results.csv'),
    Path('ablation_results.csv'),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        'Could not find ablation_results.csv. Expected at experiments/results/ablation_results.csv '
        'or repository root. Run the ablation runner first.'
    )

df = pd.read_csv(csv_path)
print(f'Loaded: {csv_path} | rows={len(df)} | cols={len(df.columns)}')
display(df.head())

In [ ]:
# Normalize/derive expected columns used by Sections 5.2-5.5 plots
if 'mean_quality_score' not in df.columns and 'quality_score' in df.columns:
    df['mean_quality_score'] = df['quality_score']

if 'n_hypotheses' not in df.columns:
    df['n_hypotheses'] = 0
if 'n_valid_hypotheses' not in df.columns:
    df['n_valid_hypotheses'] = 0

if 'hvr' not in df.columns:
    denom = df['n_hypotheses'].replace(0, np.nan)
    df['hvr'] = (df['n_valid_hypotheses'] / denom).fillna(0.0)

if 'drift_detection_latency' not in df.columns:
    df['drift_detection_latency'] = np.nan

if 'tamper_detection_rate' not in df.columns:
    # Defaults to documented target when explicit metric isn't logged in CSV
    df['tamper_detection_rate'] = np.where(df['variant'] == 'V4_no_provenance', 0.0, 1.0)

if 'provenance_completeness' not in df.columns:
    df['provenance_completeness'] = np.where(df['variant'] == 'V4_no_provenance', 0.0, 1.0)

if 'chain_verification_ms' not in df.columns:
    df['chain_verification_ms'] = np.nan

if 'runtime_s' not in df.columns:
    raise ValueError('CSV must include runtime_s from ablation runner.')

summary = df.groupby(['variant', 'drift_after'], as_index=False).agg({
    'mean_quality_score': 'mean',
    'hvr': 'mean',
    'n_hypotheses': 'mean',
    'n_valid_hypotheses': 'mean',
    'runtime_s': ['mean', 'std'],
    'provenance_completeness': 'mean',
    'tamper_detection_rate': 'mean',
    'chain_verification_ms': 'mean',
})
summary.columns = ['_'.join(c).strip('_') for c in summary.columns.to_flat_index()]
display(summary.head())

## Chart 1 (Section 5.2): Quality Score vs Drift Level
Line chart of mean quality score across variants and drift levels.

In [ ]:
plot1 = summary[['variant', 'drift_after', 'mean_quality_score_mean']].copy()
plot1 = plot1.sort_values(['variant', 'drift_after'])

fig, ax = plt.subplots(figsize=(11, 6))
sns.lineplot(
    data=plot1,
    x='drift_after',
    y='mean_quality_score_mean',
    hue='variant',
    style='variant',
    markers=True,
    dashes=False,
    linewidth=2.2,
    ax=ax
)
ax.set_title('RQ1: Mean Quality Score by Drift Level and Variant')
ax.set_xlabel('Drift Injection Batch (drift_after)')
ax.set_ylabel('Mean Quality Score')
ax.set_ylim(0, 1.02)
ax.legend(title='Variant', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
plt.show()

## Chart 2 (Section 5.3): Hypothesis Validity Rate by Drift Level
Bar chart for HVR with annotation of average generated/valid hypotheses.

In [ ]:
plot2 = summary[['variant', 'drift_after', 'hvr_mean', 'n_hypotheses_mean', 'n_valid_hypotheses_mean']].copy()
plot2 = plot2.sort_values(['drift_after', 'variant'])

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=plot2, x='drift_after', y='hvr_mean', hue='variant', ax=ax)
ax.set_title('RQ2: Hypothesis Validity Rate (HVR) by Drift Level and Variant')
ax.set_xlabel('Drift Injection Batch (drift_after)')
ax.set_ylabel('HVR (Valid / Total Hypotheses)')
ax.set_ylim(0, 1.05)
ax.legend(title='Variant', bbox_to_anchor=(1.02, 1), loc='upper left')

# Annotate bars with generated/valid counts for interpretability
for patch, (_, row) in zip(ax.patches, plot2.iterrows()):
    height = patch.get_height()
    ax.annotate(
        f"g={row['n_hypotheses_mean']:.1f}\nv={row['n_valid_hypotheses_mean']:.1f}",
        (patch.get_x() + patch.get_width() / 2, height),
        ha='center', va='bottom', fontsize=8, rotation=0,
        xytext=(0, 3), textcoords='offset points'
    )

fig.tight_layout()
plt.show()

## Chart 3 (Section 5.4): Provenance Integrity Metrics
Comparison of provenance completeness and tamper detection rate by variant, with chain verification time overlay when available.

In [ ]:
prov = df.groupby('variant', as_index=False).agg({
    'provenance_completeness': 'mean',
    'tamper_detection_rate': 'mean',
    'chain_verification_ms': 'mean'
})

prov_long = prov.melt(
    id_vars='variant',
    value_vars=['provenance_completeness', 'tamper_detection_rate'],
    var_name='metric',
    value_name='value'
)
metric_labels = {
    'provenance_completeness': 'Completeness',
    'tamper_detection_rate': 'Tamper Detection Rate'
}
prov_long['metric'] = prov_long['metric'].map(metric_labels)

fig, ax1 = plt.subplots(figsize=(11, 6))
sns.barplot(data=prov_long, x='variant', y='value', hue='metric', ax=ax1)
ax1.set_title('RQ3: Provenance Integrity by Variant')
ax1.set_xlabel('Variant')
ax1.set_ylabel('Rate')
ax1.set_ylim(0, 1.05)
ax1.legend(title='Metric', loc='upper left')

if prov['chain_verification_ms'].notna().any():
    ax2 = ax1.twinx()
    ax2.plot(prov['variant'], prov['chain_verification_ms'], color='black', marker='o', linewidth=2, label='Verification ms')
    ax2.set_ylabel('Chain Verification Time (ms)')
    ax2.legend(loc='upper right')

fig.tight_layout()
plt.show()

## Chart 4 (Section 5.5): Runtime Overhead vs Static Baseline
Bar chart of mean runtime per variant with overhead percentage relative to V5_static.

In [ ]:
runtime = df.groupby('variant', as_index=False)['runtime_s'].mean().sort_values('runtime_s', ascending=False)

baseline_row = runtime.loc[runtime['variant'] == 'V5_static', 'runtime_s']
if baseline_row.empty:
    baseline = runtime['runtime_s'].min()
else:
    baseline = float(baseline_row.iloc[0])

runtime['overhead_pct'] = ((runtime['runtime_s'] - baseline) / baseline) * 100.0

fig, ax = plt.subplots(figsize=(11, 6))
palette = ['#5B8FF9' if v != 'V5_static' else '#5AD8A6' for v in runtime['variant']]
sns.barplot(data=runtime, x='variant', y='runtime_s', palette=palette, ax=ax)
ax.set_title('Runtime Comparison and Overhead vs Static Baseline')
ax.set_xlabel('Variant')
ax.set_ylabel('Mean Runtime (s)')

for p, (_, row) in zip(ax.patches, runtime.iterrows()):
    ax.annotate(
        f"{row['overhead_pct']:+.1f}%",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha='center', va='bottom', fontsize=10,
        xytext=(0, 3), textcoords='offset points'
    )

fig.tight_layout()
plt.show()

In [ ]:
# Optional: save figures for thesis/paper
out_dir = Path('experiments/results/figures')
out_dir.mkdir(parents=True, exist_ok=True)
print(f'Figure output directory ready: {out_dir.resolve()}')